# Geospatial Research Radar - 01: morning scan

Run this notebook each morning. It queries open scholarly sources, merges duplicate records, scores them, filters previously seen items, and writes a dated JSON/CSV result file.

**JupyterLite note:** remote calls use the browser fetch stack, so each API must permit browser cross-origin requests. If one source is blocked or rate-limited, the notebook continues with the others and records the error.

In [ ]:
from pathlib import Path
import json
from radar_core import (
    DEFAULT_CONFIG, deep_copy_jsonable, load_json, save_json,
    run_scan, export_csv, top_summary, update_seen
)

config = load_json("radar_config.json", default=deep_copy_jsonable(DEFAULT_CONFIG))
INCLUDE_PREVIOUSLY_SEEN = False
MARK_AS_SEEN_AFTER_SUCCESS = True

print("Lookback:", config["lookback_days"], "days")
print("Queries:", len(config["query_families"]))
print("Watched people:", ", ".join(config["watch_people"]))

## Run the scan

The first run may return a larger baseline. Later runs are quieter because `seen_items.json` suppresses records already surfaced.

In [ ]:
result = await run_scan(
    config,
    include_seen=INCLUDE_PREVIOUSLY_SEEN,
    seen_path="seen_items.json",
)

print(f"Raw records:      {result['total_raw']}")
print(f"After dedupe:     {result['total_deduped']}")
print(f"Above threshold:  {result['total_scored']}")
print(f"New to you:       {len(result['items'])}")

if result['errors']:
    print("
Source warnings (scan still completed):")
    for err in result['errors'][:12]:
        print("-", err)

print("
TOP RESULTS
")
print(top_summary(result['items'], n=15) or "No new high-signal items in this window.")

## Inspect score explanations

This cell prints why the top items scored well, including the method/open-data/code terms that were detected and the automatically suggested MVP test.

In [ ]:
for i, x in enumerate(result['items'][:10], 1):
    print("=" * 88)
    print(f"{i}. PRIORITY {x['priority_score']} | INTEREST {x['interest_score']} | MVP {x['prototype_score']}")
    print(x['title'])
    print("Authors:", ", ".join(x.get('authors', [])[:6]))
    print("Watch:", x.get('watch_person') or "-")
    print("Open-data signals:", ", ".join(x.get('open_data_hits', [])[:10]) or "-")
    print("Implementation signals:", ", ".join(x.get('implementation_hits', [])[:10]) or "-")
    print("Heavy-compute signals:", ", ".join(x.get('heavy_compute_hits', [])[:10]) or "-")
    print("MVP idea:", x['mvp_hint'])
    print("Link:", x.get('url', ''))

## Save the morning result

The dated JSON is the canonical record. The CSV is convenient for sorting or accumulating a personal literature log.

In [ ]:
run_date = result["run_date"]
json_path = f"radar_results_{run_date}.json"
csv_path = f"radar_results_{run_date}.csv"

save_json({k:v for k,v in result.items() if k != "seen_state"}, json_path)
export_csv(result["items"], csv_path)
print("Saved:", json_path)
print("Saved:", csv_path)

if MARK_AS_SEEN_AFTER_SUCCESS:
    update_seen(result["items"], result["seen_state"], "seen_items.json")
    print("Updated seen_items.json")

### Re-running the same day

If you want to see the same items again, set `INCLUDE_PREVIOUSLY_SEEN = True` at the top and rerun. The dated result file is saved before the seen-state update, so your morning snapshot remains available.